# Planning, reflection, and the verification ladder

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 16 §16.3–16.5 · type: walkthrough*

**The promise:** layer planning and self-checking on top of ReAct, and rank exit conditions so the agent
stops on *checkable* evidence — not vibes.


## 🧠 Why this matters

ReAct (16-01) is adaptive but **myopic** — on a long task it wanders. Three patterns fix the three ways it
wanders. **Plan-and-execute** (§16.3) decomposes the task up front so the agent doesn't lose the thread.
**Reflection** (§16.4) catches a bad draft before you ship it. And the **verification ladder** (§16.4, the
chapter's 🔑 key idea) ranks your exit conditions so the loop stops on something *trustworthy*.

The judgment running through all three: **prefer checkable over vibes.** Hard verification (run the tests)
beats a grounded critique (a judge with sources) beats bare self-reflection (the model re-reading itself).
Every notch up that ladder makes "retry a few times" into an actual reliability mechanism.


## Objectives & prerequisites

**By the end you can:**
- Decompose a task into 3–7 concrete, *checkable* steps with a `Plan` schema, and **replan** when one goes stale.
- Rewrite a vague step into a verifiable one (task-decomposition drill).
- Add a **capped** reflection cycle (critique → revise → `ACCEPT`).
- Rank exit conditions on the **verification ladder** and pick the strongest one available.
- Route heterogeneous traffic with a cheap classifier that **fails safe**.

**Run first / prereqs:**
- `16-01-react-and-interleaved-thinking` (the loop each step runs).
- Ch 15 (structured outputs — the `Plan` schema) · Ch 14 (keeping notes small).
- Runs **free and offline** in `MOCK=1`. No key needed.


## Setup

Same `MOCK` discipline as 16-01: **`MOCK=1` (default)** returns canned plans, critiques, and route labels —
deterministic, free, offline. `COMPANION_MOCK=0` (plus `ANTHROPIC_API_KEY`) hits the live API: a planner call
plus a few short executor / critique / route calls.


In [ ]:
import os
import random
from typing import Literal

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from pydantic import BaseModel, Field  # pydantic is in requirements.txt

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"
MODEL = "claude-opus-4-8"
ROUTER_MODEL = "claude-haiku-4-5"  # cheap model for the router (§16.5)

random.seed(16)

if MOCK:
    print("MOCK mode: canned plans/critiques/labels, offline, free.")
else:
    assert os.getenv("ANTHROPIC_API_KEY"), "MOCK=0 needs ANTHROPIC_API_KEY in your env"
    import anthropic
    client = anthropic.Anthropic()
    print(f"LIVE mode: {MODEL} + {ROUTER_MODEL}. A planner call plus a few short calls (~cents).")


## Plan-and-execute: decompose, then run each step

When a task is long or expensive to redo, add a **planning phase** (§16.3). A planner call decomposes the task
into ordered, checkable steps; an executor then runs each step as a short, bounded ReAct run. We model the
`Plan` as a Pydantic schema (Ch 15) so the structure is validated, not hoped for. `notes` stay small (Ch 14).


In [ ]:
class Plan(BaseModel):
    steps: list[str] = Field(description="each step: one concrete, verifiable action")

TASK = "Write a Python function median(nums) and make sure it is correct."

# Canned planner output for MOCK; live mode would call a schema-validated `structured(...)` (Ch 15).
_MOCK_PLAN = Plan(steps=[
    "Write median(nums) handling odd and even lengths.",
    "Write 3 unit tests: odd length, even length, single element.",
    "Run the tests and confirm they pass.",
])

def make_plan(task: str) -> Plan:
    if MOCK:
        return _MOCK_PLAN
    # live: a schema-validated model call (see Ch 15's `structured`); kept terse here.
    raise NotImplementedError("wire to your Ch 15 structured() helper for MOCK=0")

plan = make_plan(TASK)
for i, s in enumerate(plan.steps, 1):
    print(f"{i}. {s}")


## ⚠️ Pitfall: plans go stale on contact with reality

A plan is a guess made before any step has run. The moment a step's result contradicts a later step, the rest
of the plan is wrong — and an agent that marches on regardless wastes the whole run. Production planners
**replan**: after a step (or a failure), revise the plan against what actually happened. Keep the cadence
**coarse** — replanning every step doubles your cost for marginal benefit.


In [ ]:
def replan(task: str, plan: Plan, done: list[str], surprise: str) -> Plan:
    """Coarse-cadence replan: only when a result invalidates remaining steps."""
    if MOCK:
        # The 'even length' step revealed a tie-breaking ambiguity; insert a decision step.
        remaining = [s for s in plan.steps if s not in done]
        return Plan(steps=done + ["Decide even-length tie rule: average the two middle values."] + remaining)
    raise NotImplementedError("wire to structured() for MOCK=0")

done = [plan.steps[0]]
surprise = "Even-length median is ambiguous: average the two middles, or pick one?"
plan2 = replan(TASK, plan, done, surprise)
print("AFTER REPLAN:")
for i, s in enumerate(plan2.steps, 1):
    print(f"{i}. {s}")


## Task-decomposition drill: vague → verifiable

The skill underneath planning (§16.3): a good step is **concrete, independently verifiable, and roughly
equal-sized**. "Research the market" is not a step; "list the top five competitors with pricing pages" is.
Run the drill — for each vague step, write the verifiable rewrite, then check it against a simple rule.


In [ ]:
VAGUE = [
    "Research the market.",
    "Make the function robust.",
    "Improve performance.",
]

REWRITES = {
    "Research the market.": "List the top five competitors, each with a link to its pricing page.",
    "Make the function robust.": "Add tests for empty input and non-numeric input; both must raise ValueError.",
    "Improve performance.": "Reduce median() from O(n log n) sort to O(n) selection; benchmark on n=10_000.",
}

def is_verifiable(step: str) -> bool:
    """Cheap heuristic: a checkable step names a number, an artifact, or a concrete check."""
    cues = ("list", "test", "raise", "benchmark", "link", "five", "empty", "o(n)")
    return any(c in step.lower() for c in cues) and any(ch.isdigit() or ch == 'O' for ch in step) or \
        sum(c in step.lower() for c in cues) >= 2

for v in VAGUE:
    r = REWRITES[v]
    print(f"vague     : {v}\n  verifiable: {r}\n  passes check? {is_verifiable(r)}\n")


## Reflection: critique → revise → ACCEPT (capped)

The cheapest quality win in agentic systems: **don't accept the first draft** (§16.4). Reflection adds an
evaluate-and-revise cycle — the model critiques the draft against the requirements, and the critique drives a
revision. It helps because *critiquing is easier than generating*. **Cap the rounds**: self-critique can
oscillate or "fix" what was fine.


In [ ]:
# A deliberately flawed draft: median() that breaks on even-length input.
FLAWED = "def median(nums):\n    s = sorted(nums)\n    return s[len(s) // 2]\n"

_MOCK_CRITIQUES = [
    "Defect: even-length lists return the upper-middle element, not the average of the two middles.",
    "ACCEPT",  # second round: the revision is correct
]
_MOCK_REVISION = (
    "def median(nums):\n"
    "    s = sorted(nums)\n"
    "    mid = len(s) // 2\n"
    "    if len(s) % 2:\n"
    "        return s[mid]\n"
    "    return (s[mid - 1] + s[mid]) / 2\n"
)

def ask_critique(draft: str, rnd: int) -> str:
    if MOCK:
        return _MOCK_CRITIQUES[min(rnd, len(_MOCK_CRITIQUES) - 1)]
    raise NotImplementedError("wire to client.messages.create for MOCK=0")

def ask_revise(draft: str, critique: str) -> str:
    if MOCK:
        return _MOCK_REVISION
    raise NotImplementedError("wire to client.messages.create for MOCK=0")

def reflect_loop(draft: str, max_rounds: int = 2) -> str:
    for rnd in range(max_rounds):
        critique = ask_critique(draft, rnd)
        print(f"round {rnd}: {critique}")
        if critique.strip() == "ACCEPT":
            break
        draft = ask_revise(draft, critique)
    return draft


### 🔮 Predict

Before running: will self-critique **improve** this deliberately-flawed `median`? And once it's correct, will a
second round leave it alone — or risk "fixing" something that was already fine? Predict the number of rounds and
whether the final code is correct.


In [ ]:
final_code = reflect_loop(FLAWED, max_rounds=2)
print("\nFINAL DRAFT:\n" + final_code)


**What you just saw.** Round 0 caught the even-length defect; round 1 returned `ACCEPT` and stopped. The cap is
load-bearing: without it, a model that *shares its own blind spots* can oscillate — re-critiquing correct code,
"fixing" it back into a bug. Self-critique is cheap quality, but it is not ground truth. Which is the next idea.


## 🔑 The verification ladder

The key idea of the chapter. **Rank your checks** and use the strongest one available:

1. **Hard verification** — run the tests, execute the SQL, validate the schema, compile the code. *Ground truth.*
2. **Grounded critique** — a judge model **with the source documents in hand**.
3. **Bare self-reflection** — the model re-reading itself (what we just did).

Every notch up makes the loop's exit condition more trustworthy. Below, "hard verification" is **real, not
narrated**: we actually run unit tests against the revised `median` and let the *test result* — not the model —
decide whether to exit.


In [ ]:
import unittest

def hard_verify(code_str: str) -> tuple[bool, str]:
    """Top rung: execute the code and run real tests. The result is ground truth."""
    ns: dict = {}
    try:
        exec(code_str, ns)  # trusted, locally-authored code only
    except Exception as e:  # noqa: BLE001 - report any failure to the caller
        return False, f"did not compile/exec: {e}"
    median = ns.get("median")
    cases = [([3, 1, 2], 2), ([4, 1, 2, 3], 2.5), ([5], 5)]
    for nums, expected in cases:
        got = median(nums)
        if got != expected:
            return False, f"median({nums}) == {got}, expected {expected}"
    return True, "all 3 tests passed"

ok_flawed, msg_flawed = hard_verify(FLAWED)
ok_fixed, msg_fixed = hard_verify(final_code)
print(f"flawed draft -> verified={ok_flawed}: {msg_flawed}")
print(f"fixed  draft -> verified={ok_fixed}: {msg_fixed}")


**The point:** the flawed draft *fails a test*; the fixed draft *passes*. That boolean is a trustworthy exit
condition in a way "the model said ACCEPT" never is. When ground truth exists, climb to it — reserve reflection
for the genuinely uncheckable.


## Router: cheapest reliable classifier, fail safe (§16.5)

Not every request deserves the same machinery. A **router** is a cheap, fast classification step at the front
door that sends each request down the right path — a small model over a *fixed label set*, falling safe to a
human on any unknown label. The router should be the cheapest reliable component you can build.


In [ ]:
ROUTES = ["faq", "docs_qa", "agent_task", "handoff_human"]

# Canned classifier for MOCK; live mode uses the cheap ROUTER_MODEL.
_MOCK_LABELS = {
    "what are your support hours?": "faq",
    "summarize the attached contract": "docs_qa",
    "book me a flight to lisbon": "agent_task",
    "i'm furious, cancel everything": "???",  # model returns an off-list label -> must fail safe
}

def route(query: str) -> str:
    if MOCK:
        label = _MOCK_LABELS.get(query.lower().strip(), "???")
    else:
        resp = client.messages.create(
            model=ROUTER_MODEL, max_tokens=16,
            system=f"Classify the request into one of {ROUTES}. Reply with the label only.",
            messages=[{"role": "user", "content": query}],
        )
        label = resp.content[0].text.strip()
    return label if label in ROUTES else "handoff_human"  # fail safe

for q in _MOCK_LABELS:
    print(f"{route(q):>13}  <-  {q}")


Notice the last row: the model returned an off-list label, and the router **fell back to `handoff_human`**
rather than guessing. Failing toward the safe path is the whole discipline — a misroute to a human is recoverable;
a confident misroute into an autonomous agent may not be.


## 🎯 Senior lens

**Code-orchestrator vs. model-orchestrator.** An orchestrator decomposes a request and dispatches the parts.
When the decomposition is **known in advance**, make the orchestrator *code* — deterministic, cheap, testable
(our `plan` list, run by a `for` loop). When it **genuinely varies per request**, make it a *model* — flexible,
adaptive, costlier. Reach for the model only when the work truly can't be enumerated up front; most product
decompositions can. That exact distinction — code that drives models vs. a model that drives code — is what
Chapter 17 develops into full multi-agent systems.


## Recap

- **Plan-and-execute** decomposes a long task into checkable steps; **replan** coarsely when a result goes stale.
- Good steps are **concrete, independently verifiable, equal-sized** — "list the top five competitors," not "research the market."
- **Reflection** is cheap quality (critique → revise) but shares the model's blind spots — **cap the rounds**.
- The **verification ladder**: hard verification > grounded critique > bare self-reflection. Climb as high as ground truth allows.
- A **router** is the cheapest reliable classifier over a fixed label set, and it **fails safe** to a human.
- **Code-orchestrator** for known decompositions, **model-orchestrator** for varying ones (→ Ch 17).


## Exercises

1. **Trigger a replan.** Add a step whose result *invalidates* a later step (e.g., a benchmark shows the O(n)
   rewrite is slower for small n). Predict the revised plan, then produce it.
2. **Make reflection oscillate.** Change `_MOCK_CRITIQUES` so round 1 critiques the *correct* code and "fixes" it
   back into a bug. Show why the round cap is what saves you — and why hard verification would have caught it.
3. **Climb the ladder.** Replace the `median` example with one where no tests are possible (a prose summary).
   Which rung can you reach? Implement a grounded critique (judge + source) instead of bare self-reflection.
4. **Harden the router.** Add a 5th route and a query the model labels ambiguously. Confirm it still fails safe;
   then add a confidence threshold below which it also hands off.


In [ ]:
# Exercise 1: introduce a result that invalidates a later step and replan.


In [ ]:
# Exercise 2: make the reflection loop oscillate, then show hard verification catching it.


In [ ]:
# Exercise 3: a no-tests task; implement a grounded critique (judge + source) as the exit condition.


In [ ]:
# Exercise 4: add a route + an ambiguous query; confirm fail-safe and add a confidence threshold.


## Next

- **Next notebook:** [`16-03-runbudget-and-termination-guards.ipynb`](./16-03-runbudget-and-termination-guards.ipynb) —
  every loop above contains a model deciding when to stop; bound them so none can run away.
- **Previous:** [`16-01-react-and-interleaved-thinking.ipynb`](./16-01-react-and-interleaved-thinking.ipynb).
- **Blueprint:** the router/orchestrator sketch foreshadows
  [`../../../blueprints/multi-agent-supervisor/`](../../../blueprints/multi-agent-supervisor/) (Ch 17); the
  hardened loop lives in [`../../../blueprints/agent-loop/`](../../../blueprints/agent-loop/).
- **Book:** see §16.3–16.5.
